# Stationarity & Trend Tests for Hydrological Time Series

Demonstrates the five non-parametric tests in `reef_tools.stats.stationarity`:

| Test | Purpose |
|---|---|
| `pettitt_test` | Step change detection |
| `mann_kendall` | Monotonic trend + Theil-Sen slope |
| `rank_sum_test` | Pre/post median comparison |
| `median_crossing_test` | Serial independence via median crossings |
| `rank_difference_test` | Serial independence via von Neumann ratio |

All tests are non-parametric (no normality assumption) and operate on annual flow totals.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# If running in Colab, install reef-tools first
import sys
if 'google.colab' in sys.modules:
    !pip install -q reef-tools

from reef_tools.stats import (
    pettitt_test,
    mann_kendall,
    rank_sum_test,
    median_crossing_test,
    rank_difference_test,
)

rng = np.random.default_rng(42)

## 1. Synthetic examples

Create three synthetic 50-year annual flow series:
- **Stationary**: random normal (no trend, no shift)
- **Trend**: steady +2%/yr decline
- **Step change**: regime shift at year 25 (−60% flow)

In [ ]:
years = np.arange(1970, 2020)
n = len(years)

# Stationary series
stationary = rng.normal(100, 20, n)

# Declining trend (2%/yr)
trend = 100 * (1 - 0.02 * np.arange(n)) + rng.normal(0, 8, n)

# Step change at year 25
step = np.where(np.arange(n) < 25,
                rng.normal(120, 15, n),
                rng.normal(50, 10, n))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, data, title in zip(axes, [stationary, trend, step],
                            ["Stationary", "Trend (-2%/yr)", "Step change (-60%)"]):
    ax.plot(years, data, 'o-', markersize=4)
    ax.set_title(title)
    ax.set_ylabel("Annual flow (ML/yr)")
plt.tight_layout()
plt.show()

## 2. Pettitt test — step change detection

In [ ]:
for name, data in [("Stationary", stationary), ("Trend", trend), ("Step", step)]:
    pt = pettitt_test(data)
    print(f"{name:>12s}: K={pt['K']:>6.0f}, p={pt['p_value']:.4f}, "
          f"change_idx={pt.get('change_idx', '-')}, "
          f"significant={pt['significant']}")

## 3. Mann-Kendall — monotonic trend

In [ ]:
for name, data in [("Stationary", stationary), ("Trend", trend), ("Step", step)]:
    mk = mann_kendall(data)
    print(f"{name:>12s}: tau={mk['tau']:+.3f}, p={mk['p_value']:.4f}, "
          f"slope={mk['slope']:.2f}/yr, direction={mk['direction']}")

## 4. Rank-sum — pre/post median comparison

Split at the Pettitt-detected change point and test whether medians differ.

In [ ]:
for name, data in [("Stationary", stationary), ("Trend", trend), ("Step", step)]:
    pt = pettitt_test(data)
    ci = pt["change_idx"]
    rs = rank_sum_test(data[:ci], data[ci:])
    print(f"{name:>12s}: median {rs['pre_median']:.0f} -> {rs['post_median']:.0f} "
          f"({rs['delta_pct']:+.1f}%), p={rs['p_value']:.4f}")

## 5. Median crossing & rank-difference — serial independence

In [ ]:
for name, data in [("Stationary", stationary), ("Trend", trend), ("Step", step)]:
    mc = median_crossing_test(data)
    rd = rank_difference_test(data)
    print(f"{name:>12s}: crossings={mc['n_crossings']}/{mc['expected']:.0f} "
          f"(p={mc['p_value']:.4f}, {mc['description']}), "
          f"RVN={rd['RVN']:.3f} ({rd['description']})")

## 6. Real example — Lerderderg River (HRS 231213)

Load annual flow totals from the repo and run the full test suite.  
The Lerderderg has a well-documented ~70% flow reduction after 1996.

In [ ]:
# Load annual flows directly from GitHub
DATA_URL = (
    "https://raw.githubusercontent.com/frbennett/reef-tools/main/"
    "docs/examples/data/lerderderg_annual.csv"
)
df = pd.read_csv(DATA_URL)
years = df["year"].values
annual = df["flow_ML"].values

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(years, annual, width=0.8, color='steelblue', edgecolor='white')
ax.axvline(1996, color='red', linestyle='--', linewidth=2,
           label='Detected change (1996)')
ax.set_ylabel('Annual flow (ML/yr)')
ax.set_title('Lerderderg River — Annual Flow (HRS 231213)')
ax.legend()
plt.tight_layout()
plt.show()

# Run all tests
print(f"Record: {len(years)} years ({years[0]}-{years[-1]})")
print(f"Mean: {annual.mean():.0f} ML/yr, Median: {np.median(annual):.0f} ML/yr\n")

pt = pettitt_test(annual)
mk = mann_kendall(annual)
mc = median_crossing_test(annual)
rd = rank_difference_test(annual)

print(f"Pettitt (step change): K={pt['K']:.0f}, p={pt['p_value']:.4f}, "
      f"change year={years[pt['change_idx']]}, significant={pt['significant']}")

rs = rank_sum_test(annual[:pt['change_idx']], annual[pt['change_idx']:])
print(f"Rank-sum  (medians):    {rs['pre_median']:.0f} -> {rs['post_median']:.0f} ML/yr "
      f"({rs['delta_pct']:+.1f}%), p={rs['p_value']:.4f}")

print(f"Mann-Kendall (trend):   tau={mk['tau']:+.3f}, p={mk['p_value']:.4f}, "
      f"slope={mk['slope']:.2f} ML/yr/yr, direction={mk['direction']}")

print(f"Median crossing:        {mc['n_crossings']}/{mc['expected']:.0f} expected, "
      f"p={mc['p_value']:.4f}, {mc['description']}")

print(f"Rank-difference:        RVN={rd['RVN']:.3f}, "
      f"p={rd['p_value']:.4f}, {rd['description']}")

## Summary

The Pettitt test correctly identifies the 1996 step change (K=672, p=0.0002).
The rank-sum confirms a 70% median flow reduction (p=0.0001).
Mann-Kendall detects a declining trend (-1.4%/yr) superimposed on the step.
The rank-difference von Neumann ratio (RVN=1.43) confirms strong serial
persistence — adjacent years are much more similar than random, a direct
consequence of the two flat flow regimes separated by the 1996 shift.